In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_23.pickle

In [ ]:
%%PandasProfile
### cell 23 ###

def grab_subset_of_data_35(df, question_of_interest):
    # Select only columns containing the question, strip prefix, drop rows all-NA
    subset = df.filter(like=question_of_interest)
    mapping = {col: col.split('-')[-1].lstrip() for col in subset.columns}
    return subset.rename(columns=mapping).dropna(how='all')


def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest, include_2017=None):
    # Prepare year labels and corresponding source dataframes
    years = ['2018', '2019', '2020', '2021', '2022']
    dfs = [responses_df_2018_cell10,
           responses_df_2019_cell10,
           responses_df_2020,
           responses_df_2021,
           responses_df_2022_cell10]
    if include_2017 is not None:
        years.insert(0, '2017')
        dfs.insert(0, responses_df_2017)

    # Grab subset, rename and tag with year
    processed = [
        grab_subset_of_data_35(src, question_of_interest).assign(year=yr)
        for src, yr in zip(dfs, years)
    ]

    df_combined = pd.concat(processed, ignore_index=True)
    df_combined_counts = df_combined.groupby('year').count().reset_index()
    return df_combined, df_combined_counts


def convert_df_of_counts_to_percentages_35(df, df_counts):
    # Compute percentage of respondents per year selecting each option
    counts = df_counts.set_index('year').astype(int)
    totals = df['year'].value_counts()
    pct = counts.div(totals, axis=0) * 100
    pct.index.name = 'year'
    return pct.reset_index()


# --- main pipeline ---
question_of_interest_cell35 = 'What programming languages do you use on a regular basis?'
language_df_combined, language_df_combined_counts = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_35(
        question_of_interest_cell35
    )
)
language_df_combined_percentages = convert_df_of_counts_to_percentages_35(
    language_df_combined,
    language_df_combined_counts
)

# Select and reshape for plotting
cols = ['Python', 'SQL', 'C++', 'C', 'R', 'Java', 'Javascript', 'Other']
df_cell35 = (
    language_df_combined_percentages
    .loc[:, ['year'] + cols]
    .melt(id_vars='year', value_vars=cols)
    .sort_values(['year', 'value'])
    .rename(columns={'variable': ''})
)
df_cell35.info()